In [3]:
import os
from dotenv import load_dotenv

# 텍스트를 문맥 단위로 분할하는 공식 splitter
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 텍스트를 숫자로 변환해 주는 OpenAI 임베딩 클래스
from langchain_openai import OpenAIEmbeddings

# 메모리 상에서 작동하는 벡터 저장소
from langchain_core.vectorstores import InMemoryVectorStore

In [4]:
load_dotenv()

True

In [5]:
# 1단계 - 문서 수집: 학교규정 원본 문서를 수집했다고 가정함.
raw_text = """[스파르타대학교 수강신청 및 성적 장학 규정]
제1조(신청 학점): 학생은 학기당 최소 12학점에서 최대 19학점까지 수강 신청할 수 있습니다.
제2조(초과 신청): 직전 학기 평점 평균이 4.0 이상인 학생은 다음 학기에 최대 21학점까지 신청 가능합니다.
제3조(성적 장학금): 직전 학기 15학점 이상을 이수하고 평점 평균 3.5 이상을 취득한 학생 중 선발합니다.
단, F 학점이 포함된 과목이 있는 경우 성적 장학금 지급 대상에서 제외됩니다."""

# 2단계 - 문서 분할: 문맥 단절을 방지하기 위한 겹침 구간(overlap)을 둠, 자르는 분할기 설정
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=90, # 한 조각(청크)의 최대 글자 수를 90자로 제한합니다.
    chunk_overlap=20 # 앞뒤 조각이 20자씩 겹치도록 설정하여 문맥 유실을 방지합니다.
)

# 원본 텍스트를 text_splitter 설정에 따라 여러 개의 Doucument 객체 리스트로 분할
split_docs = text_splitter.create_documents([raw_text])

# 분할 결과물 확인 : 청크 개수 및 내용 출력
print(f"총 {len(split_docs)}개의 청크로 분할되었습니다.\n")
for i, doc in enumerate(split_docs):
    print(f"[청크 {i+1}]: {doc.page_content}\n")

총 4개의 청크로 분할되었습니다.

[청크 1]: [스파르타대학교 수강신청 및 성적 장학 규정]
제1조(신청 학점): 학생은 학기당 최소 12학점에서 최대 19학점까지 수강 신청할 수 있습니다.

[청크 2]: 제2조(초과 신청): 직전 학기 평점 평균이 4.0 이상인 학생은 다음 학기에 최대 21학점까지 신청 가능합니다.

[청크 3]: 제3조(성적 장학금): 직전 학기 15학점 이상을 이수하고 평점 평균 3.5 이상을 취득한 학생 중 선발합니다.

[청크 4]: 단, F 학점이 포함된 과목이 있는 경우 성적 장학금 지급 대상에서 제외됩니다.



In [6]:
# 3단계 - 임베딩 및 저장: 분할된 조각을 숫자로 변환 -> 메모리 벡터 저장소에 적재
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vector_store = InMemoryVectorStore.from_documents(
    documents=split_docs,
    embedding=embeddings
)

print("인덱싱 완료: 문서가 임베딩되어 벡터 저장소에 안전하게 보관되었습니다.")

인덱싱 완료: 문서가 임베딩되어 벡터 저장소에 안전하게 보관되었습니다.


In [ ]:
# [임베딩 결과 들여다보기]
# 임베딩: 문장의 "뜻"을 숫자 목록(벡터)으로 바꾸는 것
#   - 지도에서 위치를 (위도, 경도)로 나타내듯,
#     문장의 뜻을 숫자 1536개짜리 좌표로 나타냄 (text-embedding-3-small 기준)
#   - 뜻이 비슷한 문장일수록 좌표도 서로 가까움
#       ex) 질문: "장학금 받으려면 몇 학점?" 과 "제3조(성적 장학금): 15학점 이상..."은 뜻이 비슷해서 좌표도 비슷함: 둘이 가깝다고 판단.
#   - 숫자 하나하나의 의미는 사람도 모름 → 중요한 건 "벡터끼리 얼마나 가까운가"
#
# RAG 검색 흐름
#   1) 문서 청크를 전부 벡터로 바꿔 저장 (3단계에서 한 일)
#   2) 질문도 같은 방식으로 벡터로 바꿈
#   3) 질문 벡터와 가장 가까운 청크를 찾아 AI에게 참고자료로 전달

# vector_store.store: InMemoryVectorStore 내부 저장 공간 (딕셔너리)
#   key   = 청크 id
#   value = {"text": 원문, "vector": 임베딩 벡터, "metadata": 부가정보}
for i, (doc_id, item) in enumerate(vector_store.store.items()):
    vec = item["vector"]                      # 이 청크의 임베딩 벡터 (숫자 1536개 리스트)
    print(f"[청크 {i+1}] id: {doc_id}")
    print(f"  텍스트  : {item['text']}")       # 원래 문장
    print(f"  벡터 차원: {len(vec)}")           # 숫자가 몇 개인지 → 1536
    # vec[:5]        : 1536개 중 앞의 5개만 잘라냄 (전부 찍으면 화면이 도배되니까 맛보기)
    # round(v, 4)    : 소수점 4자리까지 반올림 (0.012345 → 0.0123)
    # [... for v in ...] : 5개를 하나씩 반올림해서 다시 리스트로 모음
    print(f"  앞 5개 값: {[round(v, 4) for v in vec[:5]]} ...\n")


[청크 1] id: 803fef84-93d6-4857-bb39-226b95c008fc
  텍스트  : [스파르타대학교 수강신청 및 성적 장학 규정]
제1조(신청 학점): 학생은 학기당 최소 12학점에서 최대 19학점까지 수강 신청할 수 있습니다.
  벡터 차원: 1536
  앞 5개 값: [-0.0195, 0.0219, -0.0098, 0.0142, 0.0551] ...

[청크 2] id: d46c0a66-ed7b-46bc-ae10-6c572b32bd68
  텍스트  : 제2조(초과 신청): 직전 학기 평점 평균이 4.0 이상인 학생은 다음 학기에 최대 21학점까지 신청 가능합니다.
  벡터 차원: 1536
  앞 5개 값: [-0.034, -0.0007, -0.0151, 0.0076, 0.066] ...

[청크 3] id: c202f39d-9f4a-4770-bbda-cc13db57d2f7
  텍스트  : 제3조(성적 장학금): 직전 학기 15학점 이상을 이수하고 평점 평균 3.5 이상을 취득한 학생 중 선발합니다.
  벡터 차원: 1536
  앞 5개 값: [-0.0298, 0.0258, 0.0066, -0.0164, 0.0396] ...

[청크 4] id: 51e26ddc-e610-4da7-8346-76f2edae79cf
  텍스트  : 단, F 학점이 포함된 과목이 있는 경우 성적 장학금 지급 대상에서 제외됩니다.
  벡터 차원: 1536
  앞 5개 값: [-0.0121, 0.0382, -0.0187, 0.0029, 0.0195] ...



In [8]:
from langchain.chat_models import init_chat_model # 언어 모델 초기화 함수
from langchain_core.prompts import ChatPromptTemplate # 프롬프트 템플릿
from langchain_core.output_parsers import StrOutputParser # 문자열 파서

# 사용자 질문 정의
user_query = "직전 학기 성적이 4.1점인데 다음 학기에 최대 몇 학점까지 신청할 수 있나요?"

# 4단계 - 검색 : 저장소에서 질문과 가장 유사한 상위 1개의 청크(k=1)를 찾아온다.
retriever = vector_store.as_retriever(search_kwargs={"k": 1})
retrieved_results = retriever.invoke(user_query)

# 검색된 문서 조각의 내용을 텍스트 형태로 추출
context_chunk = retrieved_results[0].page_content
print("=== [4단계: 검색된 근거 청크] ===")
print(context_chunk)

=== [4단계: 검색된 근거 청크] ===
제2조(초과 신청): 직전 학기 평점 평균이 4.0 이상인 학생은 다음 학기에 최대 21학점까지 신청 가능합니다.


In [13]:
# 5단계 - 생성 : 모델 초기화 + 사실 기반 답변을 위해 temperature=0 설정
llm = init_chat_model(
    model_provider="openai",
    model="gpt-6-luna",
    api_key=os.getenv("OPENAI_API_KEY2")
)

# 검색된 문맥 (context)와 사용자의 질문(question) 결합하는 프롬프트 구성
prompt_template = ChatPromptTemplate.from_messages([
    ("system", "제공된 [참고 문서]의 내용에만 근거하여 정확하게 답변하되, [참고 문서]에 없는 내용은 모른다라고 답변하세요.\n\n[참고 문서]:\n{context}"),
    ("user", "{question}")
])

# RAG 체인 조립: 프롬프트 -> 모델 -> 문자열 파서
pipeline_chain = prompt_template | llm | StrOutputParser()

# 문맥과 질문을 전달하여 최종 답변을 생성
final_response = pipeline_chain.invoke({
    "context": context_chunk,
    "question": user_query
})

print("\n=== [5단계: 최종 생성된 답변] ===")
print(final_response)


=== [5단계: 최종 생성된 답변] ===
다음 학기에 최대 21학점까지 신청할 수 있습니다.
